# Micromouse — maze photo → drive commands

Turns a single overhead photo of the maze into a wall-accurate map, a shortest path,
and the `f`/`l`/`r` string that `chainMovement()` in `task3ChainingMovements.hpp` executes.

**Pipeline**

| stage | what happens |
|---|---|
| 1 | find the rig in the photo and flatten the perspective |
| 2 | locate the cyan wall clips → infer the grid size and spacing |
| 3 | one homography: photo → exact `N·CELL_PX` maze square |
| 4 | illumination-flattened wall mask |
| 5 | score each of the `2·N·(N+1)` cell edges → boolean wall map |
| 6 | walls → graph, BFS shortest path, flood-fill distances |
| 7 | path → `f`/`l`/`r` command string, exported for the firmware |

Nothing is hard-coded to this photo: the grid size, the maze corners and the cell
pitch are all measured from the image.

In [ ]:
import math
import textwrap
import heapq
from collections import deque

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

# ----------------------------------------------------------------- tunables
PHOTO       = "fri3.jpg"

CELL_PX     = 100          # px per maze cell in the rectified image
COARSE_PX   = 900          # size of the first (rig) rectification
CELL_SIZE_MM = 180         # physical size of one cell in mm (spec 1.2)

CLIP_HUE    = (78, 110)    # OpenCV hue window for the cyan wall clips
CLIP_SAT    = 70
CLIP_VAL    = 80
CLIP_AREA   = 25           # min blob area (px) to count as a clip
CLIP_TOL    = 25           # px: clips closer than this share a grid line

WALL_LEVEL  = 220          # flattened-gray below this = wall pixel. 200 was
                           # under-detecting: several real walls (e.g. between
                           # (3,7) and (4,7)) sit only ~15% darker than the floor
                           # because the camera sees their bright top edge rather
                           # than a side face. 220 scores those a full 1.00 and
                           # gives the fewest borderline edges overall.
EDGE_HALF   = 22           # px band either side of a cell edge when scoring
EDGE_TRIM   = 0.22         # ignore this fraction at each end (posts/corners)
WALL_THR    = 0.50         # fraction of the edge that must be dark

# ------------------------------------------------- robot / obstacle geometry
# Numbers from the assignment brief:
#   1.1  "The robot must fit within a cylinder 150 mm wide"  -> radius <= 75 mm
#   1.2  cells are 180 x 180 mm
#   4.2  "All the obstacles will be cylindrical with a diameter of 100mm"
#
# ROBOT_RADIUS_MM is a HARD FLOOR for planning: inflating by less than the
# robot's real radius produces a route the robot physically cannot fit through,
# which is exactly the failure this notebook exists to prevent.
#
# 50 mm = our measured robot (100 mm across), comfortably inside the 75 mm legal
# limit. Measure to the FURTHEST PROTRUDING POINT -- lidar mounts, wheel faces,
# and any overhang count, not just the chassis. The safety margin below absorbs
# small measurement error, but it cannot absorb a forgotten sensor bracket.
ROBOT_RADIUS_MM  = 50      # measured; spec allows up to 75
SAFETY_MARGIN_MM = 20      # extra padding; relaxed automatically if a zone is too tight
OBSTACLE_DIA_MM  = 100     # spec 4.2 -- used to sanity-check the detected pillars

# The planner searches down from ROBOT_RADIUS_MM + SAFETY_MARGIN_MM to
# ROBOT_RADIUS_MM and reports the clearance it actually achieved. It never plans
# below ROBOT_RADIUS_MM; if nothing fits there, the zone is genuinely impassable
# for this robot and it raises rather than emitting a route that cannot be driven.
CLEARANCE_STEP_MM = 1

# Soft clearance preference. The two constants above are a hard constraint: they
# decide what is drivable at all. These decide *where inside* the drivable space
# to run, which is a separate question, and the one that governs how many
# waypoints come out.
#
# A* on its own returns the shortest route, which grazes the inflated obstacle
# boundary. smooth_path then cannot help but emit a chord per pixel there: around
# a convex pillar, any chord between two boundary points cuts through the pillar,
# so no long chord is ever free. Inflating harder does not fix that -- it moves
# the same hugging onto a larger circle, over a longer arc, for *more* waypoints.
#
# Penalising proximity instead bows the route onto the middle of the gap, where
# there is slack on both sides and long chords verify free. A handful of
# waypoints then covers the zone, which is what the robot wants: every waypoint
# is a profiled turn paid for in settling time and heading error.
CLEARANCE_PREF_MM = 45     # bow away from obstacles until this much slack, then stop caring
CLEARANCE_WEIGHT  = 3.0    # how much extra path length that slack is worth

# smooth_path works on the pixel grid, so it can leave a one- or two-pixel tail
# on the end of a route. At this scale that is under 2 mm, and the direction of
# such a segment is quantisation noise rather than geometry -- but the exporter
# still reads a heading off it, and the correction turn into Path B then has to
# undo that heading. Segments shorter than this are dropped before any turn is
# measured. Keep it above MISSION_MIN_DRIVE (5 mm) in the firmware, which would
# skip the drive while still executing the bogus turn that precedes it.
MIN_SEGMENT_MM = 10

# Start pose is (row, col, direction) and the goal is (row, col), both zero-indexed
# with row 0 at the top (north) of the photo -- the convention in the task brief.
# START = (1, 6, "N") therefore means 2nd row, 7th column, facing north.
START, START_HEADING, GOAL = (3,0), "W", (3, 7)

# Zone info (whole maze)
ZONE_ENTER  = (3, 2)
ZONE_EXIT   = (1, 6)
START_HEADING = 0.0  # 0.0 = East, 90.0 = South, -90.0 = North, 180.0 = West

# The 5x5 obstacle zone, given as its TOP-LEFT cell: the SMALLEST row and the
# SMALLEST column. The crop runs forward from here --
#     rows ZONE_START_R .. ZONE_START_R + ZONE_SIZE - 1
#     cols ZONE_START_C .. ZONE_START_C + ZONE_SIZE - 1
# -- so on a 9x9 board with ZONE_SIZE = 5 both values must be in 0..4.
#
# The comment that used to sit here said "top right corner". That was wrong, and
# reading it that way is a trap: for a zone spanning cols 2..6 you would write 6,
# which asks for cols 6..10 and runs two cells off the board.
ZONE_START_R, ZONE_START_C = 0, 2
ZONE_SIZE = 5

# The entrance and exit in ROI pixels, measured from the zone's top-left corner.
# Both must land inside the ZONE_SIZE x ZONE_SIZE crop -- Stage 4 re-derives these
# and refuses to continue if either falls outside it.
ZONE_START_PX = (
    int((ZONE_ENTER[1] - ZONE_START_C + 0.45) * CELL_PX), # X (cols)
    int((ZONE_ENTER[0] - ZONE_START_R + 0.45) * CELL_PX)  # Y (rows)
)

ZONE_GOAL_PX = (
    int((ZONE_EXIT[1] - ZONE_START_C + 0.45) * CELL_PX),  # X (cols)
    int((ZONE_EXIT[0] - ZONE_START_R + 0.45) * CELL_PX)   # Y (rows)
)

# The brief specifies the goal as (row, col) only, so the final heading is free.
# Set this to "N"/"E"/"S"/"W" to also park the robot facing a given way.
GOAL_HEADING = None

PATH_DOT = "•"

PX_PER_MM = CELL_PX / CELL_SIZE_MM

DISTANCE_TUNING_FACTOR = 0.80

plt.rcParams["figure.dpi"] = 110
print("OpenCV", cv2.__version__, "| NumPy", np.__version__)
print(f"scale: {PX_PER_MM:.3f} px/mm  |  robot radius {ROBOT_RADIUS_MM} mm "
      f"({ROBOT_RADIUS_MM * PX_PER_MM:.1f} px)  |  {OBSTACLE_DIA_MM} mm obstacle "
      f"= {OBSTACLE_DIA_MM * PX_PER_MM:.1f} px across")

## Stage 1 — find the rig and flatten it

The rig is by far the largest bright object in the frame, so: threshold → largest
connected component → fill holes → 4-point polygon. That quad is the perspective
reference. No hand-typed pixel coordinates, and it re-derives itself for a
re-shot photo.

In [ ]:
def order_quad(pts):
    """Return the 4 points ordered top-left, top-right, bottom-right, bottom-left."""
    p = np.asarray(pts, np.float32).reshape(-1, 2)
    s, d = p.sum(axis=1), p[:, 0] - p[:, 1]
    return np.float32([p[np.argmin(s)],    # tl: smallest x+y
                       p[np.argmax(d)],    # tr: largest  x-y
                       p[np.argmax(s)],    # br: largest  x+y
                       p[np.argmin(d)]])   # bl: smallest x-y


def largest_bright_blob(bgr, level):
    """Binary mask of the biggest bright region, with interior holes filled."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    mask = cv2.morphologyEx((gray > level).astype(np.uint8) * 255,
                            cv2.MORPH_CLOSE, np.ones((21, 21), np.uint8))

    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    biggest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mask = (labels == biggest).astype(np.uint8) * 255

    # flood the outside from a corner, then invert it to get the holes
    flooded = mask.copy()
    cv2.floodFill(flooded, np.zeros((h + 2, w + 2), np.uint8), (0, 0), 255)
    return mask | cv2.bitwise_not(flooded)


def find_rig_quad(bgr, level=130):
    mask = largest_bright_blob(bgr, level)
    contour = max(cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)[0], key=cv2.contourArea)

    quad = cv2.approxPolyDP(contour, 0.02 * cv2.arcLength(contour, True), True)
    if len(quad) != 4:                       # fall back if the outline is noisy
        quad = cv2.boxPoints(cv2.minAreaRect(contour))
    return order_quad(quad)


def square_transform(quad, size):
    """Homography mapping `quad` onto a `size` x `size` square."""
    return cv2.getPerspectiveTransform(
        quad, np.float32([[0, 0], [size, 0], [size, size], [0, size]]))

In [ ]:
photo = cv2.imread(PHOTO)
if photo is None:
    raise FileNotFoundError(f"could not read {PHOTO!r} - run this notebook from src/task4_code/")

rig_quad = find_rig_quad(photo)
M_coarse = square_transform(rig_quad, COARSE_PX)
coarse   = cv2.warpPerspective(photo, M_coarse, (COARSE_PX, COARSE_PX))

print("rig corners (tl, tr, br, bl):")
for name, (x, y) in zip("tl tr br bl".split(), rig_quad):
    print(f"  {name}  ({x:7.1f}, {y:7.1f})")

marked = photo.copy()
cv2.polylines(marked, [rig_quad.astype(int)], True, (0, 255, 0), 4)
for x, y in rig_quad.astype(int):
    cv2.circle(marked, (x, y), 12, (0, 0, 255), -1)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(cv2.cvtColor(marked, cv2.COLOR_BGR2RGB)); ax[0].set_title("detected rig")
ax[1].imshow(cv2.cvtColor(coarse, cv2.COLOR_BGR2RGB)); ax[1].set_title("perspective removed")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## Stage 2 — read the grid off the wall clips

Every wall is held by a pair of cyan clips, and those clips sit on cell corners.
Isolating them by hue and clustering their centres in x and y recovers the grid
lines directly from the hardware — which is far more reliable than assuming the
maze fills the board.

This maze is 9 × 9, but rows and columns are counted **independently** and
`ROWS`/`COLS` are carried separately from here on, so the size is measured rather
than asserted and a re-shot or differently-shaped board needs no code change.

In [ ]:
def find_clips(bgr):
    """Centroids of the cyan wall clips."""
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    hue, sat, val = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]

    mask = ((hue > CLIP_HUE[0]) & (hue < CLIP_HUE[1]) &
            (sat > CLIP_SAT) & (val > CLIP_VAL)).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    n, _, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    return np.array([centroids[i] for i in range(1, n)
                     if stats[i, cv2.CC_STAT_AREA] >= CLIP_AREA])


def cluster_1d(values, tol):
    """Collapse 1-D values into groups whose neighbours are within `tol`."""
    ordered = np.sort(np.asarray(values, float))
    groups = [[ordered[0]]]
    for v in ordered[1:]:
        if v - groups[-1][-1] <= tol:
            groups[-1].append(v)
        else:
            groups.append([v])
    return np.array([np.mean(g) for g in groups])


def infer_grid(clips, tol=CLIP_TOL):
    """Grid line positions (xs, ys) implied by the clip centroids."""
    return cluster_1d(clips[:, 0], tol), cluster_1d(clips[:, 1], tol)

In [ ]:
clips  = find_clips(coarse)
gx, gy = infer_grid(clips)

# Counted independently rather than assumed equal, so the size comes from the
# photo. For this board both come out as 9.
COLS, ROWS = len(gx) - 1, len(gy) - 1
if ROWS < 2 or COLS < 2:
    raise RuntimeError(f"implausible {ROWS} x {COLS} grid from {len(clips)} clips "
                       "- widen CLIP_TOL or the hue window")

print(f"{len(clips)} clips  ->  {ROWS} rows x {COLS} cols")
print("x lines:", np.round(gx, 1))
print("y lines:", np.round(gy, 1))
print("cell pitch: %.1f px (x), %.1f px (y)" % (np.diff(gx).mean(), np.diff(gy).mean()))

vis = coarse.copy()
for x in gx: cv2.line(vis, (int(x), 0), (int(x), COARSE_PX), (255, 200, 0), 2)
for y in gy: cv2.line(vis, (0, int(y)), (COARSE_PX, int(y)), (255, 200, 0), 2)
for x, y in clips.astype(int): cv2.circle(vis, (x, y), 6, (0, 0, 255), -1)

plt.figure(figsize=(7.5, 7.5))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"clips (red) and the {ROWS}x{COLS} grid they imply")
plt.axis("off"); plt.show()

## Picking the coordinates — the labelled grid

Every cell drawn on the photo with its `(row, col)` label, using the grid lines
measured back in Stage 2. Row 0 is the top of the photo and column 0 the left, the
same convention the constants at the top of the notebook use — read a pair off this
picture, put it up there, and re-run.

Same cell as in the 4.1 notebook, plus the two things 4.2 adds: the dashed yellow
square is the `5x5` obstacle course (`ZONE_START_R` / `ZONE_START_C` / `ZONE_SIZE`),
and `ZONE IN` / `ZONE OUT` are the handover points where the discrete maze legs meet
the continuous planner.

It sits here, straight after the grid is measured, rather than at the end of the
notebook: if the zone coordinates are wrong the planner further down raises before
it ever draws anything, and this is the picture you need in order to fix them.

In [ ]:
import io

from matplotlib.figure import Figure
from matplotlib.patches import Rectangle
from matplotlib import patheffects
from IPython.display import display, Image

MAX_LABELS = 2000     # a real board is ~81 cells; near this means Stage 2 misread the clips


def show_cell_labels(img, xs, ys, mark=(), zone=None, title=None):
    """Overlay the measured grid on `img` and label every cell with its (row, col).

    `xs`/`ys` are the grid lines from Stage 2, so the labels land on the same cells
    the wall scoring uses. `mark` is an optional list of (label, (r, c), colour) to
    ring, and `zone` an optional (start_row, start_col, size) obstacle-course square.

    Drawn on a bare `Figure` rather than through pyplot, so no GUI window is ever
    created and no figure is left open: this returns straight away whichever
    backend the kernel happens to be on.
    """
    rows, cols = len(ys) - 1, len(xs) - 1
    if rows < 1 or cols < 1 or rows * cols > MAX_LABELS:
        raise RuntimeError(f"{rows} x {cols} grid from the clip clustering - too many "
                           "lines to label, re-run Stage 2 and check CLIP_TOL")

    pitch = min(np.diff(xs).mean(), np.diff(ys).mean())
    outline = [patheffects.withStroke(linewidth=2.5, foreground="white")]

    fig = Figure(figsize=(9, 9 * rows / cols), dpi=plt.rcParams["figure.dpi"])
    ax = fig.add_subplot(111)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    for x in xs:
        ax.plot([x, x], [ys[0], ys[-1]], color="#1e90ff", lw=1.6)
    for y in ys:
        ax.plot([xs[0], xs[-1]], [y, y], color="#1e90ff", lw=1.6)

    # the 5x5 obstacle course, drawn before the labels so the text stays on top
    if zone is not None:
        zr, zc, zn = zone
        if zr + zn > rows or zc + zn > cols:
            raise ValueError(
                f"the {zn}x{zn} zone at (row {zr}, col {zc}) runs off a {rows}x{cols} board. "
                f"A {zn}x{zn} window only fits with ZONE_START_R in 0..{rows - zn} and "
                f"ZONE_START_C in 0..{cols - zn} -- the origin is the TOP-LEFT cell of the "
                "zone, not its centre or its far corner.")
        ax.add_patch(Rectangle((xs[zc], ys[zr]), xs[zc + zn] - xs[zc], ys[zr + zn] - ys[zr],
                               fill=False, edgecolor="#ffd21f", lw=4, linestyle="--"))
        ax.text((xs[zc] + xs[zc + zn]) / 2, ys[zr] - pitch * 0.18,
                f"{zn}x{zn} OBSTACLE ZONE", color="#ffd21f", fontsize=pitch * 0.13,
                fontweight="bold", ha="center", va="center", path_effects=outline)

    for r in range(rows):
        for c in range(cols):
            ax.text((xs[c] + xs[c + 1]) / 2, (ys[r] + ys[r + 1]) / 2, f"{r},{c}",
                    color="#ff3b1f", fontsize=pitch * 0.13, fontweight="bold",
                    ha="center", va="center", path_effects=outline)

    for name, (r, c), colour in mark:
        ax.add_patch(Rectangle((xs[c], ys[r]), xs[c + 1] - xs[c], ys[r + 1] - ys[r],
                               fill=False, edgecolor=colour, lw=3))
        ax.text((xs[c] + xs[c + 1]) / 2, ys[r] + pitch * 0.22, name, color=colour,
                fontsize=pitch * 0.11, fontweight="bold", ha="center", va="center",
                path_effects=outline)

    ax.set_title(title or f"Choose coordinates from this labelled {rows} x {cols} grid")
    ax.axis("off")

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    display(Image(data=buf.getvalue()))


show_cell_labels(
    coarse, gx, gy,
    mark=[("START",    START,      "#2ee06e"),
          ("ZONE IN",  ZONE_ENTER, "#ffd21f"),
          ("ZONE OUT", ZONE_EXIT,  "#ffd21f"),
          ("GOAL",     GOAL,       "#d740ff")],
    zone=(ZONE_START_R, ZONE_START_C, ZONE_SIZE),
)

# 4.2 uses degrees for the heading (0 = East) rather than 4.1's N/E/S/W letter.
HEADING_NAME = {0.0: "E", 90.0: "S", -90.0: "N", 180.0: "W"}

print(f"board       : {ROWS} rows x {COLS} cols")
print(f"START       : {START} facing "
      f"{HEADING_NAME.get(START_HEADING, '?')} ({START_HEADING:g} deg)")
print(f"GOAL        : {GOAL}")
print(f"obstacle zone: {ZONE_SIZE}x{ZONE_SIZE} with its top-left at "
      f"(row {ZONE_START_R}, col {ZONE_START_C}) "
      f"-> rows {ZONE_START_R}..{ZONE_START_R + ZONE_SIZE - 1}, "
      f"cols {ZONE_START_C}..{ZONE_START_C + ZONE_SIZE - 1}")

# The entrance and exit must lie inside the declared zone -- they are the handover
# points between the discrete maze legs and the continuous planner, and getting
# them wrong is a silent way to plan a route through part of the course.
for name, (r, c) in (("ZONE_ENTER", ZONE_ENTER), ("ZONE_EXIT", ZONE_EXIT)):
    inside = (ZONE_START_R <= r < ZONE_START_R + ZONE_SIZE and
              ZONE_START_C <= c < ZONE_START_C + ZONE_SIZE)
    print(f"  {name:11}{(r, c)}: {'inside the zone' if inside else 'OUTSIDE THE ZONE - fix this'}")

## Stage 3 — one homography, photo → maze square

Rather than warping twice and losing sharpness, the rig transform and the
grid-fitting transform are multiplied into a single matrix `M`. The result is a
square where **cell `(r, c)` is exactly the box `[c·CELL_PX, (c+1)·CELL_PX] ×
[r·CELL_PX, (r+1)·CELL_PX]`** — so every later step is plain array slicing.

In [ ]:
MAZE_W, MAZE_H = COLS * CELL_PX, ROWS * CELL_PX
grid_quad = np.float32([[gx[0],  gy[0]],  [gx[-1], gy[0]],
                        [gx[-1], gy[-1]], [gx[0],  gy[-1]]])

M = cv2.getPerspectiveTransform(
        grid_quad,
        np.float32([[0, 0], [MAZE_W, 0], [MAZE_W, MAZE_H], [0, MAZE_H]])) @ M_coarse
maze_img = cv2.warpPerspective(photo, M, (MAZE_W, MAZE_H))

def cell_centre(r, c):
    return int((c + 0.5) * CELL_PX), int((r + 0.5) * CELL_PX)

print(f"rectified maze: {maze_img.shape[1]} x {maze_img.shape[0]} px, {CELL_PX} px per cell")

plt.figure(figsize=(8 * COLS / max(ROWS, COLS) + 2, 8 * ROWS / max(ROWS, COLS) + 1))
plt.imshow(cv2.cvtColor(maze_img, cv2.COLOR_BGR2RGB))
plt.xticks(np.arange(0, MAZE_W + 1, CELL_PX)); plt.yticks(np.arange(0, MAZE_H + 1, CELL_PX))
plt.grid(color="yellow", alpha=.55)
plt.title("rectified maze - grid ticks are cell boundaries")
plt.show()

## Stage 4 — wall pixels

The board is lit unevenly (bright at the top, shadowed at the bottom), so a fixed
grey threshold cannot separate wall from floor across the whole image. A
morphological *closing* with a kernel wider than any wall estimates the local
floor brightness; dividing by it flattens the lighting and leaves a threshold
that works everywhere.

In [ ]:
def wall_pixels(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    # closing with a kernel wider than a wall keeps only the floor -> local brightness
    floor = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, np.ones((61, 61), np.uint8))
    flat  = cv2.divide(gray, floor, scale=255)

    mask = (flat < WALL_LEVEL).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  np.ones((3, 3), np.uint8))   # speckle
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))   # seams
    return mask


def zone_obstacles(mask, start_r, start_c, size_cells):
    """Crop the zone and return (obstacle_mask 0/1, roi_bgr).

    Normalised to 0/1 here. `wall_pixels()` returns a 0/1 mask, not 0/255, and
    the old code ran `cv2.bitwise_not()` on it -- giving 255 for free and 254
    for blocked, both non-zero, so every `== 0` collision test downstream
    silently never fired and the planner drove through the pillars.
    """
    y0, x0 = start_r * CELL_PX, start_c * CELL_PX
    y1, x1 = (start_r + size_cells) * CELL_PX, (start_c + size_cells) * CELL_PX
    return (mask[y0:y1, x0:x1] > 0).astype(np.uint8), maze_img[y0:y1, x0:x1].copy()


def zone_cell_px(name, cell):
    """(row, col) -> (ROI pixel, complaint or None).

    Reports rather than raises, so the caller can still draw the C-space picture
    before failing. A cell outside the zone maps to an ROI pixel outside the
    crop, and every downstream test then calls it 'blocked' -- which reads as
    "the course is impassable" when the real answer is "the zone constants are
    wrong", so the complaint has to name that distinction.
    """
    r, c = cell
    r_lo, r_hi = ZONE_START_R, ZONE_START_R + ZONE_SIZE - 1
    c_lo, c_hi = ZONE_START_C, ZONE_START_C + ZONE_SIZE - 1
    px = (int((c - ZONE_START_C + 0.45) * CELL_PX),
          int((r - ZONE_START_R + 0.45) * CELL_PX))

    if r_lo <= r <= r_hi and c_lo <= c <= c_hi:
        return px, None

    need_r = max(0, min(r, r_lo) if r < r_lo else r - ZONE_SIZE + 1)
    need_c = max(0, min(c, c_lo) if c < c_lo else c - ZONE_SIZE + 1)
    return px, (
        f"{name} = {cell} is OUTSIDE the declared {ZONE_SIZE}x{ZONE_SIZE} zone, which covers "
        f"rows {r_lo}..{r_hi}, cols {c_lo}..{c_hi} -> it maps to ROI pixel {px}.\n"
        f"    Either move {name}, or move the zone: to reach {cell} you would need about "
        f"ZONE_START_R, ZONE_START_C = {need_r}, {need_c}.\n"
        f"    The labelled-grid cell above draws the zone on the photo -- check it against where "
        f"the pillars actually are.")


def audit_pillars(obstacle_mask):
    """Compare the detected free-standing blobs against the 100 mm spec.

    Spec 4.2 fixes every obstacle at 100 mm diameter, which gives us a free
    correctness check on the vision: if the blobs do not come out near 100 mm,
    the threshold is wrong and the occupancy map cannot be trusted -- better to
    see that here than to discover it when the robot hits something.
    """
    n, labels, stats, cents = cv2.connectedComponentsWithStats(obstacle_mask, connectivity=8)
    h, w = obstacle_mask.shape
    pillars = []
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < 200:                       # speckle
            continue
        x, y = stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP]
        bw, bh = stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
        if x <= 1 or y <= 1 or x + bw >= w - 1 or y + bh >= h - 1:
            continue                         # touches the crop edge -> maze wall, not a pillar
        pillars.append((cents[i], bw / PX_PER_MM, bh / PX_PER_MM))

    print(f"  detected {len(pillars)} free-standing pillar(s) (spec: {OBSTACLE_DIA_MM} mm diameter)")
    for k, (c, wmm, hmm) in enumerate(pillars):
        # A cylinder photographed off-axis leans, so the blob runs tall; the
        # narrow axis is the honest estimate of its footprint.
        est = min(wmm, hmm)
        flag = "" if abs(est - OBSTACLE_DIA_MM) <= 25 else "   <-- off spec, check WALL_LEVEL"
        print(f"    pillar {k}: {wmm:5.0f} x {hmm:5.0f} mm  (footprint ~{est:3.0f} mm){flag}")
    return pillars


def build_free_space(obstacle_mask, inflate_mm):
    """Grow obstacles by `inflate_mm` and return a boolean drivable map."""
    radius_px = max(1, int(round(inflate_mm * PX_PER_MM)))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                       (2 * radius_px + 1, 2 * radius_px + 1))  # odd -> centred
    free_space = cv2.dilate(obstacle_mask, kernel) == 0
    # Seal the crop border: outside the zone is unknown, and the route is not
    # allowed to escape through the edge of the ROI.
    free_space[0, :] = free_space[-1, :] = False
    free_space[:, 0] = free_space[:, -1] = False
    return free_space


def show_cspace(obstacle_mask, roi_bgr, inflations, start_px, goal_px, path=None):
    """Photo plus the inflated no-go region at each clearance, endpoints marked.

    Drawn BEFORE planning on purpose. When the search fails this is the picture
    that says whether the obstacles have been over-grown, whether an endpoint
    has been swallowed, or whether the zone crop is in the wrong place -- so it
    must not live downstream of the thing that raises.
    """
    inflations = list(inflations)
    fig, axes = plt.subplots(1, len(inflations) + 1,
                             figsize=(5.0 * (len(inflations) + 1), 5.6))
    axes = np.atleast_1d(axes)
    axes[0].imshow(cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"zone photo\nrows {ZONE_START_R}..{ZONE_START_R + ZONE_SIZE - 1}, "
                      f"cols {ZONE_START_C}..{ZONE_START_C + ZONE_SIZE - 1}")

    h, w = obstacle_mask.shape

    def inside(p):
        return 0 <= p[0] < w and 0 <= p[1] < h

    for ax, mm in zip(axes[1:], inflations):
        free = build_free_space(obstacle_mask, mm)
        overlay = roi_bgr.copy()
        overlay[~free] = (0.45 * overlay[~free] + 0.55 * np.array([0, 0, 255])).astype(np.uint8)
        if path:
            for a, b in zip(path, path[1:]):
                cv2.line(overlay, a, b, (0, 255, 0), 3)
        ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))

        # An endpoint outside the crop cannot be drawn; say so instead of
        # silently plotting nothing, because that is the interesting case.
        states = []
        for p, colour, label in ((start_px, "#2ee06e", "in"), (goal_px, "#d740ff", "out")):
            if inside(p):
                ax.plot(*p, "o", ms=13, mfc="none", mec=colour, mew=3)
                states.append(f"{label} {'clear' if free[p[1], p[0]] else 'BLOCKED'}")
            else:
                states.append(f"{label} OFF-MAP")
        ax.set_title(f"obstacles grown {mm:.0f} mm ({round(mm * PX_PER_MM)} px)\n"
                     f"free {free.mean():.0%} | " + ", ".join(states))
    for a in axes:
        a.axis("off")
    plt.tight_layout(); plt.show()


wall_mask = wall_pixels(maze_img)
print(f"wall pixels: {wall_mask.mean():.1%} of the board")

tint = maze_img.copy(); tint[wall_mask > 0] = (0, 0, 255)
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(cv2.addWeighted(maze_img, .45, tint, .55, 0), cv2.COLOR_BGR2RGB))
plt.title("wall mask over the rectified photo"); plt.axis("off"); plt.show()

obstacle_mask, roi_img = zone_obstacles(wall_mask, ZONE_START_R, ZONE_START_C, ZONE_SIZE)

# Recomputed here rather than trusted from the tunables cell, because only here
# do we know the size of the crop they have to land in.
ZONE_START_PX, enter_problem = zone_cell_px("ZONE_ENTER", ZONE_ENTER)
ZONE_GOAL_PX,  exit_problem  = zone_cell_px("ZONE_EXIT",  ZONE_EXIT)
print(f"zone crop: {roi_img.shape[1]} x {roi_img.shape[0]} px "
      f"(rows {ZONE_START_R}..{ZONE_START_R + ZONE_SIZE - 1}, "
      f"cols {ZONE_START_C}..{ZONE_START_C + ZONE_SIZE - 1})")
print(f"ZONE_ENTER {ZONE_ENTER} -> ROI pixel {ZONE_START_PX}")
print(f"ZONE_EXIT  {ZONE_EXIT} -> ROI pixel {ZONE_GOAL_PX}")

print("\nobstacle audit against the brief:")
audit_pillars(obstacle_mask)

# The configuration space the planner will search: at the ideal clearance and at
# the bare robot radius it may fall back to. Drawn BEFORE the validation below
# raises, so a bad zone still gives you a picture to diagnose it from.
show_cspace(obstacle_mask, roi_img,
            [ROBOT_RADIUS_MM + SAFETY_MARGIN_MM, ROBOT_RADIUS_MM],
            ZONE_START_PX, ZONE_GOAL_PX)

if enter_problem or exit_problem:
    raise ValueError("\n".join(p for p in (enter_problem, exit_problem) if p))

## Stage 5 — from pixels to a wall map

This is the step the old ray-casting approach got wrong. Sampling a straight line
between two cell centres asks *"is anything dark in the way"*, which fires on the
side face of a neighbouring wall and misses walls the line clips at an angle.

Instead each **cell edge** is scored on its own. For the edge between two cells we
take a band `±EDGE_HALF` px around it, collapse it along the short axis (so a wall
counts wherever it falls inside the band — this absorbs the parallax lean of a
5 cm wall seen from a camera that is not directly above it), and ask what
fraction of the edge's length is covered.

The ends are trimmed by `EDGE_TRIM`, because the clips and any perpendicular wall
meeting at that corner would otherwise vote for a wall that is not there.

In [ ]:
def edge_score(mask, kind, r, c):
    """Covered fraction of one cell edge.

    kind 'H' -> the horizontal edge on the NORTH side of cell (r, c)
    kind 'V' -> the vertical   edge on the WEST  side of cell (r, c)
    """
    trim = int(CELL_PX * EDGE_TRIM)

    if kind == "H":
        y, x0, x1 = r * CELL_PX, c * CELL_PX, (c + 1) * CELL_PX
        band  = mask[max(0, y - EDGE_HALF):y + EDGE_HALF + 1, x0 + trim:x1 - trim]
        along = band.max(axis=0)          # per column: any wall pixel in the band?
    else:
        x, y0, y1 = c * CELL_PX, r * CELL_PX, (r + 1) * CELL_PX
        band  = mask[y0 + trim:y1 - trim, max(0, x - EDGE_HALF):x + EDGE_HALF + 1]
        along = band.max(axis=1)

    return float(along.mean()) if along.size else 0.0


def zone_interior_edges(rows, cols):
    """Masks of the H/V edges lying STRICTLY inside the obstacle course.

    Spec 4.2: "a 5*5 cell section of the maze will be replaced by an obstacle
    course". Replaced, so there are no walls in there -- only 100 mm cylinders.
    The perimeter is deliberately excluded: those edges are genuine maze walls.
    """
    hz = np.zeros((rows + 1, cols), bool)
    vz = np.zeros((rows, cols + 1), bool)
    r0, c0, n = ZONE_START_R, ZONE_START_C, ZONE_SIZE
    hz[r0 + 1:r0 + n, c0:c0 + n] = True     # horizontal edges between two zone rows
    vz[r0:r0 + n, c0 + 1:c0 + n] = True     # vertical edges between two zone cols
    return hz, vz


def build_walls(mask, rows, cols):
    """Score every edge, then threshold into two boolean wall maps.

    H[r, c] - wall on the north side of (r, c);  shape (rows+1, cols)
    V[r, c] - wall on the west  side of (r, c);  shape (rows, cols+1)
    """
    h_score = np.array([[edge_score(mask, "H", r, c) for c in range(cols)] for r in range(rows + 1)])
    v_score = np.array([[edge_score(mask, "V", r, c) for c in range(cols + 1)] for r in range(rows)])

    H, V = h_score >= WALL_THR, v_score >= WALL_THR

    # the board edge is a physical barrier even where no wall is clipped to it
    H[0, :] = H[-1, :] = True
    V[:, 0] = V[:, -1] = True

    # No maze walls can exist inside the obstacle course, so anything scored in
    # there is pillar noise -- a cylinder sitting near a cell boundary reads as a
    # short dark bar and trips the edge scorer. Clearing them removes a whole
    # class of false walls that the perimeter checks would otherwise have to
    # argue with. The zone's own perimeter is left alone: those are real walls,
    # and they carry the single entrance and exit.
    hz, vz = zone_interior_edges(rows, cols)
    cleared = int(H[hz].sum() + V[vz].sum())
    H[hz] = False
    V[vz] = False
    if cleared:
        print(f"cleared {cleared} spurious wall(s) from inside the {ZONE_SIZE}x{ZONE_SIZE} "
              "obstacle course (pillars are not walls)")
    return H, V, h_score, v_score


H, V, h_score, v_score = build_walls(wall_mask, ROWS, COLS)
print(f"walls found: {H.sum()} horizontal, {V.sum()} vertical "
      f"(of {H.size} and {V.size} possible edges)")

# Cells strictly inside the course. The discrete legs must not route through
# these: the wall map says they are open (correctly -- there are no walls), but
# the pillars are only represented in the pixel occupancy map, so a BFS shortcut
# across the zone would drive straight through an obstacle.
ZONE_INTERIOR = {(r, c)
                 for r in range(ZONE_START_R, ZONE_START_R + ZONE_SIZE)
                 for c in range(ZONE_START_C, ZONE_START_C + ZONE_SIZE)}

### Does the wall map match the photo?

Two checks, both ported from the 4.1 notebook, which had them and this one did not
— their absence is why a missing wall between `(3,7)` and `(4,7)` went unnoticed.

Scores should sit hard against 0 or 1. Anything mid-range is a judgement call worth
eyeballing, and a wall the scorer *misses* is the dangerous direction: the planner
will happily route straight through it.

In [ ]:
# Scores should sit hard against 0 or 1. Anything mid-range is a judgement call
# worth eyeballing before trusting the map.
def ambiguous(h_score, v_score, lo=0.25, hi=0.75):
    out = []
    for kind, arr in (("H", h_score), ("V", v_score)):
        for (r, c), s in np.ndenumerate(arr):
            if lo < s < hi:
                out.append((s, kind, r, c))
    return sorted(out, reverse=True)


decided = np.concatenate([h_score.ravel(), v_score.ravel()])
print(f"{np.mean((decided < 0.25) | (decided > 0.75)):.1%} of edges scored decisively\n")

flagged = ambiguous(h_score, v_score)
if flagged:
    print(f"{len(flagged)} borderline edge(s) - threshold is {WALL_THR}:")
    for s, kind, r, c in flagged:
        side = "north of" if kind == "H" else "west of"
        print(f"  {s:.2f}  {side} cell ({r}, {c})  ->  {'WALL' if s >= WALL_THR else 'open'}")
else:
    print("no borderline edges")

plt.figure(figsize=(9, 3.2))
plt.hist(decided, bins=40, color="#3b82f6")
plt.axvline(WALL_THR, color="crimson", ls="--", label=f"WALL_THR = {WALL_THR}")
plt.title("edge score distribution"); plt.xlabel("covered fraction of edge")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
def draw_wall_check(base, H, V):
    vis  = base.copy()
    rows, cols = H.shape[0] - 1, H.shape[1]
    for r in range(rows + 1):
        for c in range(cols):
            on = H[r, c]
            cv2.line(vis, (c * CELL_PX, r * CELL_PX), ((c + 1) * CELL_PX, r * CELL_PX),
                     (0, 255, 0) if on else (255, 0, 255), 4 if on else 1)
    for r in range(rows):
        for c in range(cols + 1):
            on = V[r, c]
            cv2.line(vis, (c * CELL_PX, r * CELL_PX), (c * CELL_PX, (r + 1) * CELL_PX),
                     (0, 255, 0) if on else (255, 0, 255), 4 if on else 1)
    return vis


plt.figure(figsize=(9.5 * COLS / max(ROWS, COLS) + 1, 9.5 * ROWS / max(ROWS, COLS) + 1))
plt.imshow(cv2.cvtColor(draw_wall_check(maze_img, H, V), cv2.COLOR_BGR2RGB))
plt.title("green = wall detected   magenta = open"); plt.axis("off"); plt.show()

## Stage 6 — the maze as a graph

`Maze` owns the wall maps and answers connectivity questions; `Graph` is the
explicit adjacency structure built from it. Node ids follow the existing
convention, `id = row · COLS + col`, so they read left-to-right then top-to-bottom.

Headings are ordered clockwise, which is what makes the turn from any heading to
any other a single modulo-4 subtraction — so **every** starting orientation is
handled, a 180° reversal included.

In [ ]:
HEADINGS = ["N", "E", "S", "W"]                                  # clockwise
DELTA    = {"N": (-1, 0), "E": (0, 1), "S": (1, 0), "W": (0, -1)}


class Node:
    def __init__(self, node_id, x, y):
        self.id, self.x, self.y = node_id, x, y

    def get_point(self):
        return self.x, self.y

    def __repr__(self):
        return f"Node({self.id} @ {self.x},{self.y})"


class Graph:
    def __init__(self):
        self.nodes, self.edges = {}, {}

    def add_node(self, node_id, x, y):
        self.nodes[node_id] = Node(node_id, x, y)
        self.edges.setdefault(node_id, [])

    def add_edge(self, a, b, weight=1):
        self.edges[a].append((b, weight))

    def remove_edge(self, a, b):
        # the original broke out of the loop after one iteration regardless
        self.edges[a] = [e for e in self.edges[a] if e[0] != b]

    def neighbours(self, node_id):
        return [n for n, _ in self.edges[node_id]]

    def get_edge_weight(self, a, b):
        return next((w for n, w in self.edges[a] if n == b), None)

    def __repr__(self):
        return f"Graph({len(self.nodes)} nodes, {sum(map(len, self.edges.values()))} directed edges)"


class Maze:
    """Wall maps plus the search routines that run on them."""

    def __init__(self, H, V):
        self.H, self.V = H, V
        self.rows, self.cols = H.shape[0] - 1, H.shape[1]

    @property
    def shape(self):
        return self.rows, self.cols

    def inside(self, r, c):
        return 0 <= r < self.rows and 0 <= c < self.cols

    # ---------------------------------------------------------- topology
    def wall(self, r, c, d):
        return bool({"N": self.H[r, c],     "S": self.H[r + 1, c],
                     "W": self.V[r, c],     "E": self.V[r, c + 1]}[d])

    def neighbours(self, r, c):
        for d, (dr, dc) in DELTA.items():
            nr, nc = r + dr, c + dc
            if self.inside(nr, nc) and not self.wall(r, c, d):
                yield (nr, nc), d

    def cells(self):
        for r in range(self.rows):
            for c in range(self.cols):
                yield r, c

    # id = row * cols + col, so ids read left-to-right then top-to-bottom
    def node_id(self, r, c):
        return r * self.cols + c

    def cell(self, node_id):
        return divmod(node_id, self.cols)

    def to_graph(self):
        g = Graph()
        for r, c in self.cells():
            x, y = cell_centre(r, c)
            g.add_node(self.node_id(r, c), x, y)
        for r, c in self.cells():
            for (nr, nc), _ in self.neighbours(r, c):
                g.add_edge(self.node_id(r, c), self.node_id(nr, nc), 1)
        return g

    # ---------------------------------------------------------- search
    def bfs(self, start, goal, avoid=()):
        """Fewest-cells path from start to goal, or None.

        `avoid` names cells the route may not pass through -- the obstacle-course
        interior, for the discrete legs. `start` and `goal` are always allowed,
        so a leg can still begin or end on the zone perimeter.
        """
        if start == goal:
            return [start]

        avoid = set(avoid) - {start, goal}
        came_from = {start: None}
        queue = deque([start])
        while queue:
            current = queue.popleft()
            for nb, _ in self.neighbours(*current):
                if nb in came_from or nb in avoid:
                    continue
                came_from[nb] = current
                if nb == goal:
                    path = [nb]
                    while came_from[path[-1]] is not None:
                        path.append(came_from[path[-1]])
                    return path[::-1]
                queue.append(nb)
        return None

    def flood(self, goal):
        """Classic micromouse flood fill: cell -> moves to reach `goal` (-1 = unreachable)."""
        dist = np.full((self.rows, self.cols), -1, int)
        dist[goal] = 0
        queue = deque([goal])
        while queue:
            current = queue.popleft()
            for nb, _ in self.neighbours(*current):
                if dist[nb] < 0:
                    dist[nb] = dist[current] + 1
                    queue.append(nb)
        return dist

    def components(self):
        seen, groups = set(), []
        for start in self.cells():
            if start in seen:
                continue
            queue, group = deque([start]), []
            seen.add(start)
            while queue:
                current = queue.popleft()
                group.append(current)
                for nb, _ in self.neighbours(*current):
                    if nb not in seen:
                        seen.add(nb)
                        queue.append(nb)
            groups.append(group)
        return sorted(groups, key=len, reverse=True)

    # ---------------------------------------------------------- rendering
    def ascii(self, path=(), start=None, goal=None):
        marks = {cell: PATH_DOT for cell in path}
        if start is not None: marks[start] = "S"
        if goal  is not None: marks[goal]  = "G"

        lines = []
        for r in range(self.rows + 1):
            lines.append("".join("+" + ("---" if self.H[r, c] else "   ")
                                 for c in range(self.cols)) + "+")
            if r < self.rows:
                lines.append("".join(("|" if self.V[r, c] else " ") + f" {marks.get((r, c), ' ')} "
                                     for c in range(self.cols))
                             + ("|" if self.V[r, self.cols] else " "))
        return "\n".join(lines)


# ===================================================================
# Continuous (pixel-level) planning inside the 5x5 obstacle zone
# ===================================================================
# All points here are (x, y) in ROI pixel coordinates, and `free_space` is the
# boolean C-space map: True = the robot's centre may sit here, False = collision.

SQRT2 = math.sqrt(2.0)


def is_free(free_space, x, y):
    """True only if (x, y) is inside the map AND drivable."""
    h, w = free_space.shape
    return 0 <= x < w and 0 <= y < h and bool(free_space[y, x])


def segment_is_free(free_space, p, q):
    """True if the straight line p->q stays entirely in free space.

    Sampled at half-pixel steps so a thin obstacle cannot slip between samples.
    """
    (x0, y0), (x1, y1) = p, q
    steps = max(1, int(math.hypot(x1 - x0, y1 - y0) * 2))
    for i in range(steps + 1):
        t = i / steps
        if not is_free(free_space, int(round(x0 + (x1 - x0) * t)),
                                   int(round(y0 + (y1 - y0) * t))):
            return False
    return True


def nearest_free(free_space, point, max_radius_px=25):
    """Closest drivable pixel to `point`, or None if there is none nearby.

    `max_radius_px` stays under half a cell so a nudged endpoint is still
    inside the intended cell -- the discrete Path A / Path B legs are handed
    over at cell granularity, so the route must not drift into a neighbour.
    """
    if is_free(free_space, *point):
        return point
    for radius in range(1, max_radius_px + 1):
        best, best_d = None, None
        for dy in range(-radius, radius + 1):
            for dx in range(-radius, radius + 1):
                if max(abs(dx), abs(dy)) != radius:      # ring only
                    continue
                x, y = point[0] + dx, point[1] + dy
                if is_free(free_space, x, y):
                    d = math.hypot(dx, dy)
                    if best_d is None or d < best_d:
                        best, best_d = (x, y), d
        if best is not None:
            return best
    return None


def astar_pixel_grid(free_space, start, goal):
    """8-connected A* over the C-space map. Returns a dense pixel path or None.

    Two things the earlier version got wrong:

    1. The obstacle test was `free_space[ny, nx] == 0`, but the mask it was
       handed only ever held 254 and 255, so the test never fired and the
       search happily expanded straight through obstacles. It now tests the
       boolean map directly.
    2. Diagonal steps were allowed unconditionally, so the path could squeeze
       through the corner where two obstacles touch diagonally -- a gap of zero
       width that the robot cannot physically pass. A diagonal is now only legal
       when both of its orthogonal partners are also free.
    """
    if not is_free(free_space, *start):
        raise ValueError(f"start {start} is inside an obstacle (after inflation)")
    if not is_free(free_space, *goal):
        raise ValueError(f"goal {goal} is inside an obstacle (after inflation)")

    neighbors = [(0, 1, 1.0), (1, 0, 1.0), (0, -1, 1.0), (-1, 0, 1.0),
                 (1, 1, SQRT2), (1, -1, SQRT2), (-1, 1, SQRT2), (-1, -1, SQRT2)]

    # See CLEARANCE_PREF_MM. Distance from every free pixel to the nearest
    # blocked one, turned into a step multiplier: 1.0 out in the open, rising to
    # 1 + CLEARANCE_WEIGHT hard against the boundary. It never drops below 1.0,
    # so the Euclidean heuristic below stays admissible and A* stays optimal.
    dist_px = cv2.distanceTransform(free_space.astype(np.uint8), cv2.DIST_L2, 5)
    pref_px = max(1.0, CLEARANCE_PREF_MM * PX_PER_MM)
    step_cost = 1.0 + CLEARANCE_WEIGHT * (1.0 - np.clip(dist_px / pref_px, 0.0, 1.0))

    open_set = [(0.0, start)]
    came_from = {}
    g_score = {start: 0.0}
    closed = set()

    while open_set:
        _, current = heapq.heappop(open_set)
        if current in closed:            # stale heap entry
            continue
        closed.add(current)

        if current == goal:
            path = [current]
            while path[-1] in came_from:
                path.append(came_from[path[-1]])
            return path[::-1]

        cx, cy = current
        for dx, dy, cost in neighbors:
            nx, ny = cx + dx, cy + dy
            if not is_free(free_space, nx, ny):
                continue
            # no cutting the corner between two diagonally-touching obstacles
            if dx and dy and not (is_free(free_space, nx, cy) and is_free(free_space, cx, ny)):
                continue

            tentative_g = g_score[current] + cost * step_cost[ny, nx]
            neighbor = (nx, ny)
            if tentative_g < g_score.get(neighbor, float("inf")):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score = tentative_g + math.hypot(goal[0] - nx, goal[1] - ny)
                heapq.heappush(open_set, (f_score, neighbor))
    return None


def smooth_path(free_space, pixel_path):
    """Reduce a dense path to waypoints, keeping every segment collision-free.

    This replaces the old `cv2.approxPolyDP` call. Douglas-Peucker knows nothing
    about obstacles: it only asks whether the simplified line stays within
    `epsilon` px of the original, so on a path that hugs a pillar it cheerfully
    replaces the detour around the pillar with a chord straight through it. On
    this maze it was cutting through 3 of 5 segments.

    Instead this walks the path greedily and takes the furthest point still
    reachable by an unobstructed straight line (standard "string pulling"). The
    result is at least as short, and every segment is drivable by construction.
    """
    if len(pixel_path) < 2:
        return list(pixel_path)

    waypoints = [pixel_path[0]]
    i = 0
    while i < len(pixel_path) - 1:
        j = len(pixel_path) - 1
        while j > i + 1 and not segment_is_free(free_space, pixel_path[i], pixel_path[j]):
            j -= 1
        waypoints.append(pixel_path[j])
        i = j
    return waypoints


def verify_waypoints(free_space, waypoints):
    """Hard check that the exported route never crosses an obstacle."""
    bad = [(k, waypoints[k], waypoints[k + 1])
           for k in range(len(waypoints) - 1)
           if not segment_is_free(free_space, waypoints[k], waypoints[k + 1])]
    if bad:
        detail = ", ".join(f"segment {k}: {p}->{q}" for k, p, q in bad)
        raise RuntimeError(f"{len(bad)} waypoint segment(s) pass through an obstacle -- {detail}")
    return True


def plan_with_clearance(obstacle_mask, start_px, goal_px,
                        robot_radius_mm=ROBOT_RADIUS_MM,
                        margin_mm=SAFETY_MARGIN_MM,
                        step_mm=CLEARANCE_STEP_MM):
    """Plan at the largest clearance that still gets through the zone.

    A 180 mm corridor gives a 150 mm robot only 15 mm per side, so the ideal
    radius + full safety margin is often simply not achievable -- at 85 mm even
    the exit cell centre is swallowed. Rather than silently under-inflating (the
    old behaviour, which is what let the route cross a pillar), this walks the
    clearance down from the ideal and reports what it actually got.

    `robot_radius_mm` is a hard floor. Below it the route would be one the robot
    physically cannot fit through, so the function raises instead.

    Returns (free_space, dense_path, start, goal, inflate_mm).
    """
    h, w = obstacle_mask.shape
    for name, pt in (("entrance", start_px), ("exit", goal_px)):
        if not (0 <= pt[0] < w and 0 <= pt[1] < h):
            raise ValueError(
                f"the zone {name} maps to ROI pixel {pt}, which is outside the "
                f"{w}x{h} px crop. That is a zone-coordinate problem, not a clearance one: "
                f"the cell lies outside rows {ZONE_START_R}..{ZONE_START_R + ZONE_SIZE - 1} / "
                f"cols {ZONE_START_C}..{ZONE_START_C + ZONE_SIZE - 1}. Fix ZONE_ENTER / ZONE_EXIT "
                "or ZONE_START_R / ZONE_START_C and re-run from the labelled-grid cell.")

    ideal = robot_radius_mm + margin_mm
    tried = []
    mm = float(ideal)

    while mm >= robot_radius_mm - 1e-9:
        free = build_free_space(obstacle_mask, mm)
        s = nearest_free(free, start_px)
        g = nearest_free(free, goal_px)
        if s is None or g is None:
            stuck = "entrance" if s is None else "exit"
            tried.append((mm, f"{stuck} has no free pixel within half a cell"))
        else:
            path = astar_pixel_grid(free, s, g)
            if path is not None:
                return free, path, s, g, mm
            tried.append((mm, "endpoints clear but no route between them"))
        mm -= step_mm

    # Distinguish "the endpoints are buried" from "the middle is blocked": they
    # have completely different fixes, and calling both "impassable" sent us
    # hunting for a smaller robot when a zone coordinate was simply wrong.
    endpoint_trouble = all("free pixel" in why for _, why in tried)
    bare = build_free_space(obstacle_mask, robot_radius_mm)
    detail = "; ".join(f"{m:.0f} mm: {why}" for m, why in tried[:3])

    if endpoint_trouble:
        buried = [(n, p) for n, p in (("ZONE_ENTER", start_px), ("ZONE_EXIT", goal_px))
                  if not bare[p[1], p[0]]]
        who = ", ".join(f"{n} at ROI pixel {p}" for n, p in buried) or "an endpoint"
        raise RuntimeError(
            f"{who} sits inside an obstacle even at the bare robot radius of {robot_radius_mm} mm "
            f"({detail}).\n"
            "    That normally means the zone crop is offset from the real obstacle course, so a "
            "pillar is landing on top of the entrance or exit cell. Check the C-space picture in "
            "Stage 4 against the labelled grid before touching ROBOT_RADIUS_MM.")

    raise RuntimeError(
        f"The endpoints are clear but no route connects them, even at the bare robot radius of "
        f"{robot_radius_mm} mm ({detail}).\n"
        f"    Free space at {robot_radius_mm} mm is {bare.mean():.0%} of the zone. If the C-space "
        "picture in Stage 4 shows the pillars swollen into one another, the wall mask is "
        "over-detecting (raise WALL_LEVEL); if it looks right, the course really is blocked for a "
        f"{2 * robot_radius_mm} mm-wide robot.")


def drop_micro_segments(free_space, pts, min_mm=MIN_SEGMENT_MM):
    """Drop waypoints whose incoming segment is too short to carry a heading.

    Obstacle-aware, for exactly the reason smooth_path does not use
    approxPolyDP: dropping a waypoint replaces two segments with the chord
    between their endpoints, and on a path that hugs a pillar that chord can cut
    straight through it. An interior point is therefore only dropped once the
    segment that would replace it has been checked free.

    The tail is a separate case -- there is no following segment to check, so it
    is simply truncated. That can only shorten the route, and every surviving
    segment is one smooth_path already proved drivable. The cost is that the
    route can stop up to min_mm short of the planned exit, which is well inside
    what the wall trim on the first corridor move takes back.
    """
    if len(pts) < 3:
        return list(pts)

    min_px = min_mm * PX_PER_MM

    def short(a, b):
        return math.hypot(b[0] - a[0], b[1] - a[1]) < min_px

    kept = [pts[0]]
    i = 1
    while i < len(pts) - 1:
        if short(kept[-1], pts[i]) and segment_is_free(free_space, kept[-1], pts[i + 1]):
            i += 1                     # drop it: the chord past it is clear
            continue
        kept.append(pts[i])
        i += 1

    if not short(kept[-1], pts[-1]):
        kept.append(pts[-1])

    # Pathological case: everything collapsed. Better to export the unfiltered
    # route, which at least verifies, than a one-point route with no segments.
    return kept if len(kept) >= 2 else list(pts)


print("Solving continuous pixel grid... (this may take a few seconds)")

free_space_mask, dense_path, start_px, goal_px, used_mm = plan_with_clearance(
    obstacle_mask, ZONE_START_PX, ZONE_GOAL_PX)

ideal_mm = ROBOT_RADIUS_MM + SAFETY_MARGIN_MM
print(f"planned with {used_mm:.0f} mm clearance "
      f"(robot {ROBOT_RADIUS_MM} mm + margin {SAFETY_MARGIN_MM} mm = {ideal_mm} mm ideal)")
if used_mm < ideal_mm:
    print(f"  NOTE: safety margin reduced to {used_mm - ROBOT_RADIUS_MM:.0f} mm -- "
          f"the zone is too tight for the full {SAFETY_MARGIN_MM} mm.")
if start_px != ZONE_START_PX:
    print(f"  note: start nudged {ZONE_START_PX} -> {start_px} to clear the inflated obstacles")
if goal_px != ZONE_GOAL_PX:
    print(f"  note: goal  nudged {ZONE_GOAL_PX} -> {goal_px} to clear the inflated obstacles")

raw_waypoints = smooth_path(free_space_mask, dense_path)
waypoints = drop_micro_segments(free_space_mask, raw_waypoints)
verify_waypoints(free_space_mask, waypoints)   # raises rather than exporting a bad route

print(f"Path found! Reduced from {len(dense_path)} pixels to {len(waypoints)} waypoints.")
if len(waypoints) != len(raw_waypoints):
    print(f"  dropped {len(raw_waypoints) - len(waypoints)} segment(s) under {MIN_SEGMENT_MM} mm")
print(f"All waypoint segments verified collision-free at {used_mm:.0f} mm clearance.")

In [ ]:
maze  = Maze(H, V)
graph = maze.to_graph()
print(graph)

## Display 5x5 path

In [ ]:
# Visually verify. The inflated no-go region is drawn underneath the path, so
# "the green line never touches red" is the same check the planner just made --
# and this is the image to show the demonstrator for the 4.2 occupancy-map mark.
vis_img = roi_img.copy()
vis_img[~free_space_mask] = (0.6 * vis_img[~free_space_mask]
                             + 0.4 * np.array([0, 0, 255])).astype(np.uint8)

for i in range(len(dense_path) - 1):                       # dense A* path
    cv2.line(vis_img, dense_path[i], dense_path[i + 1], (255, 0, 0), 2)
for i in range(len(waypoints) - 1):                        # simplified route
    cv2.line(vis_img, waypoints[i], waypoints[i + 1], (0, 255, 0), 4)
    cv2.circle(vis_img, waypoints[i], 6, (255, 255, 255), -1)
cv2.circle(vis_img, waypoints[-1], 6, (255, 255, 255), -1)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(free_space_mask, cmap="gray")
ax[0].set_title(f"Occupancy map (white = drivable)\nobstacles grown by {used_mm:.0f} mm")
ax[1].imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
ax[1].set_title(f"Trajectory at {used_mm:.0f} mm clearance\n"
                f"red = no-go | blue = A* dense | green = waypoints")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

print(f"robot footprint {2 * ROBOT_RADIUS_MM} mm wide in {CELL_SIZE_MM} mm cells "
      f"-> {(CELL_SIZE_MM - 2 * ROBOT_RADIUS_MM) / 2:.0f} mm nominal clearance per side in a corridor")

## Stage 7 — commands for the robot

Get commands from both 4.1, and the specific turns and movement for 4.2.

In [ ]:
TURN = {0: [], 1: ["r"], 2: ["r", "r"], 3: ["l"]}     # clockwise quarter-turns -> commands


def turn_to(heading, facing):
    """Commands that rotate from `heading` to `facing` (180 deg becomes 'rr')."""
    return TURN[(HEADINGS.index(facing) - HEADINGS.index(heading)) % 4]


def path_to_commands(path, heading, final_heading=None):
    """Cell path + starting heading -> (commands, final heading).

    Any of N/E/S/W works as the starting heading; the first move simply gets
    however many turns it needs, including a 'rr' reversal.

    `final_heading` is optional - the brief gives the goal as (row, col) with no
    direction, so by default the robot stops facing whichever way it arrived.
    Pass one to append the turns that park it facing that way.
    """
    commands = []
    for (r0, c0), (r1, c1) in zip(path, path[1:]):
        facing = {(-1, 0): "N", (0, 1): "E", (1, 0): "S", (0, -1): "W"}[(r1 - r0, c1 - c0)]
        commands += turn_to(heading, facing)
        commands.append("f")
        heading = facing

    if final_heading is not None:
        commands += turn_to(heading, final_heading)
        heading = final_heading
    return commands, heading


def replay(maze, start, commands, heading):
    """Independent check: walk the command string, refusing to pass through a wall.

    Returns the (cell, heading) the robot ends up in.
    """
    r, c = start
    for cmd in commands:
        if cmd == "f":
            assert not maze.wall(r, c, heading), f"command drives into a wall at ({r},{c})"
            dr, dc = DELTA[heading]
            r, c = r + dr, c + dc
        else:
            step = 1 if cmd == "r" else -1
            heading = HEADINGS[(HEADINGS.index(heading) + step) % 4]
    return (r, c), heading

def generate_relative_commands(waypoints, initial_heading_deg):
    commands = []
    current_heading = initial_heading_deg

    for i in range(len(waypoints) - 1):
        x1, y1 = waypoints[i]
        x2, y2 = waypoints[i+1]

        dist_px = math.hypot(x2 - x1, y2 - y1)
        raw_dist_mm = dist_px * (CELL_SIZE_MM / CELL_PX)
        tuned_dist_mm = raw_dist_mm * DISTANCE_TUNING_FACTOR

        # FIX: Removed the negative sign on (y2 - y1).
        # Now moving down (+Y) yields a positive angle (+90 for South),
        # perfectly matching your discrete heading map.
        angle_rad = math.atan2(y2 - y1, x2 - x1)
        angle_deg = math.degrees(angle_rad)

        turn_deg = angle_deg - current_heading
        turn_deg = (turn_deg + 180) % 360 - 180  # Normalize to [-180, 180]

        commands.append({
            "turn_deg": round(turn_deg, 1),
            "drive_mm": round(tuned_dist_mm, 1)
        })
        current_heading = angle_deg

    return commands

firmware_commands = generate_relative_commands(waypoints, 0)

def generate_discrete_commands(path_nodes, start_heading_deg):
    """
    Converts a list of (row, col) tuples into a string of discrete commands.
    Returns: (command_string, final_heading_deg)
    """
    cmds = ""
    current_heading = start_heading_deg

    for i in range(1, len(path_nodes)):
        r1, c1 = path_nodes[i - 1]
        r2, c2 = path_nodes[i]

        # Determine required absolute heading to reach the next cell
        if r2 < r1:   req_heading = -90.0  # North
        elif r2 > r1: req_heading = 90.0   # South
        elif c2 > c1: req_heading = 0.0    # East
        elif c2 < c1: req_heading = 180.0  # West
        else: continue

        # Calculate relative turn required
        turn = req_heading - current_heading
        turn = (turn + 180) % 360 - 180

        # Append turn commands
        if turn == 90.0:   cmds += "r"
        elif turn == -90.0:  cmds += "l"
        elif turn == 180.0 or turn == -180.0: cmds += "ll" # U-turn

        # Append forward command
        cmds += "f"
        current_heading = req_heading

    return cmds, current_heading



firmware_commands = generate_relative_commands(waypoints, -90)

In [ ]:
# puts all the commands in one array
def build_master_mission(start_node, zone_enter, zone_exit, goal_node, initial_heading, waypoints, maze_obj):
    unified_commands = []

    # ---------------------------------------------------------
    # PHASE 1: Path A (Start -> Zone Entrance)
    # ---------------------------------------------------------
    path_a_nodes = maze_obj.bfs(start_node, zone_enter, avoid=ZONE_INTERIOR)

    if path_a_nodes is None:
        raise ValueError(f"PATH BLOCKED: No route found from Start {start_node} to Zone Enter {zone_enter}. Check your coordinates!")

    path_a_str, heading_at_entrance = generate_discrete_commands(path_a_nodes, initial_heading)

    for char in path_a_str:
        unified_commands.append(f"{{'{char}', 0.0}}")

    # ---------------------------------------------------------
    # PHASE 2: Continuous Obstacle Zone
    # ---------------------------------------------------------
    zone_commands = generate_relative_commands(waypoints, heading_at_entrance)

    final_continuous_heading = heading_at_entrance

    for cmd in zone_commands:
        if abs(cmd["turn_deg"]) > 1.0:
            unified_commands.append(f"{{'T', {-cmd['turn_deg']}}}")
            final_continuous_heading += cmd["turn_deg"]

        if cmd["drive_mm"] > 1.0:
            unified_commands.append(f"{{'D', {cmd['drive_mm']}}}")

    # Normalize final continuous heading to [-180, 180]
    final_continuous_heading = (final_continuous_heading + 180) % 360 - 180

    # ---------------------------------------------------------
    # PHASE 3: Path B Transition & Execution
    # ---------------------------------------------------------
    path_b_nodes = maze_obj.bfs(zone_exit, goal_node, avoid=ZONE_INTERIOR)

    if path_b_nodes is None:
        raise ValueError(f"PATH BLOCKED: No route found from Zone Exit {zone_exit} to Goal {goal_node}. Check your coordinates!")

    if len(path_b_nodes) > 1:
        # 1. Peek at the first step of Path B to find the required orthogonal heading
        r1, c1 = path_b_nodes[0]
        r2, c2 = path_b_nodes[1]

        if r2 < r1:   req_heading = -90.0  # North
        elif r2 > r1: req_heading = 90.0   # South
        elif c2 > c1: req_heading = 0.0    # East
        elif c2 < c1: req_heading = 180.0  # West
        else: req_heading = final_continuous_heading

        # 2. Calculate the exact continuous turn to correct the robot's alignment
        correction_turn = req_heading - final_continuous_heading
        correction_turn = (correction_turn + 180) % 360 - 180

        # 3. Append the Continuous Correction Turn if necessary
        if abs(correction_turn) > 0.1:
            unified_commands.append(f"{{'T', {-round(correction_turn, 1)}}}")

        # 4. Generate Path B discrete commands
        # Because we pass `req_heading` as the starting heading, generate_discrete_commands
        # will see that the turn difference is 0, safely skip emitting any 'r' or 'l' for
        # the first cell, and directly emit the 'f' command to drive forward.
        path_b_str, final_heading = generate_discrete_commands(path_b_nodes, req_heading)

        for char in path_b_str:
            unified_commands.append(f"{{'{char}', 0.0}}")

    return unified_commands

Full Route Plot

In [ ]:
def plot_master_mission(maze_img, start_node, zone_enter, zone_exit, goal_node, waypoints, zone_start_r, zone_start_c):
    # Get the base image and convert to RGB for Matplotlib
    rgb = cv2.cvtColor(maze_img, cv2.COLOR_BGR2RGB)
    overlay = rgb.copy()

    # ---------------------------------------------------------
    # 1. Get Coordinates for Path A (Discrete)
    # ---------------------------------------------------------
    path_a_nodes = maze.bfs(start_node, zone_enter, avoid=ZONE_INTERIOR)
    if path_a_nodes is None:
        raise ValueError(
            f"no discrete route from START {start_node} to ZONE_ENTER {zone_enter} without "
            f"crossing the obstacle course. Check both against the labelled grid: START must sit "
            f"outside the zone and ZONE_ENTER on its perimeter, reachable from START.")
    pts_a = [cell_centre(r, c) for r, c in path_a_nodes]

    # ---------------------------------------------------------
    # 2. Get Global Coordinates for Continuous Waypoints
    # ---------------------------------------------------------
    # Calculate the pixel offset of the 5x5 ROI's top-left corner
    offset_x = zone_start_c * CELL_PX
    offset_y = zone_start_r * CELL_PX

    # Shift local waypoint coordinates back into global image coordinates
    pts_w = [(x + offset_x, y + offset_y) for x, y in waypoints]

    # ---------------------------------------------------------
    # 3. Get Coordinates for Path B (Discrete)
    # ---------------------------------------------------------
    path_b_nodes = maze.bfs(zone_exit, goal_node, avoid=ZONE_INTERIOR)
    if path_b_nodes is None:
        raise ValueError(
            f"no discrete route from ZONE_EXIT {zone_exit} to GOAL {goal_node} without crossing "
            f"the obstacle course. Check both against the labelled grid.")
    pts_b = [cell_centre(r, c) for r, c in path_b_nodes]

    # ---------------------------------------------------------
    # 4. Draw the Paths
    # ---------------------------------------------------------
    # Draw Path A (Orange)
    for p, q in zip(pts_a, pts_a[1:]):
        cv2.line(overlay, p, q, (255, 165, 0), 10)

    # Draw Continuous Waypoints (Red)
    for p, q in zip(pts_w, pts_w[1:]):
        cv2.line(overlay, p, q, (255, 0, 0), 10)
        cv2.circle(overlay, p, 7, (255, 255, 255), -1) # White dots at waypoint nodes
    cv2.circle(overlay, pts_w[-1], 7, (255, 255, 255), -1)

    # Draw Path B (Magenta)
    for p, q in zip(pts_b, pts_b[1:]):
        cv2.line(overlay, p, q, (255, 0, 255), 10)

    # ---------------------------------------------------------
    # 5. Connect the Transition Points (White thin lines)
    # ---------------------------------------------------------
    cv2.line(overlay, pts_a[-1], pts_w[0], (255, 255, 255), 4)
    cv2.line(overlay, pts_w[-1], pts_b[0], (255, 255, 255), 4)

    # ---------------------------------------------------------
    # 6. Mark the Key Nodes
    # ---------------------------------------------------------
    cv2.circle(overlay, pts_a[0], 22, (46, 220, 110), 6)  # Start (Green Outline)
    cv2.circle(overlay, pts_a[-1], 22, (255, 255, 0), 6)  # Zone Enter (Yellow Outline)
    cv2.circle(overlay, pts_b[0], 22, (255, 255, 0), 6)   # Zone Exit (Yellow Outline)
    cv2.circle(overlay, pts_b[-1], 22, (255, 64, 64), 6)  # Goal (Red Outline)

    # Blend the overlay with the original image so we can still see the maze texture
    out = cv2.addWeighted(rgb, 0.45, overlay, 0.55, 0)

    # ---------------------------------------------------------
    # 7. Render Plot
    # ---------------------------------------------------------
    plt.figure(figsize=(6, 6))
    plt.imshow(out)
    plt.title("Full Mission Trajectory\nOrange = Path A | Red = Continuous Zone | Magenta = Path B", fontsize=14, pad=15)
    plt.axis("off")
    plt.show()

# Execute the visualization using your global variables
plot_master_mission(
    maze_img=maze_img,
    start_node=START,
    zone_enter=ZONE_ENTER,
    zone_exit=ZONE_EXIT,
    goal_node=GOAL,
    waypoints=waypoints,
    zone_start_r=ZONE_START_R,
    zone_start_c=ZONE_START_C
)

## Stage 10 — hand it to the firmware

Writes `maze_route.h` next to the notebook: drop it in `src/task4_code/` and call
`chainMovement(mouse, MAZE_ROUTE)`.

In [ ]:
# 5x5 specific print for debugging
print("\n--- CONTINUOUS COMMANDS EXPORT ---")
print("Command Sequence (Relative turn, then drive straight):")
for i, cmd in enumerate(firmware_commands):
    print(f"Step {i+1}: Turn {cmd['turn_deg']:>6.1f}° | Drive {cmd['drive_mm']:>5.1f} mm")

# C-Header string formatting
c_array = ", ".join([f"{{{c['turn_deg']}, {c['drive_mm']}}}" for c in firmware_commands])

c_header = f"""#pragma once
// Generated by task4_code/image_processing.ipynb for Task 4.2
// Start position in 5x5 zone: {ZONE_START_PX}
// Goal position in 5x5 zone: {ZONE_GOAL_PX}

struct WaypointCmd {{
    float turn_deg;
    float drive_mm;
}};

constexpr int NUM_WAYPOINTS = {len(firmware_commands)};
constexpr WaypointCmd OBSTACLE_ROUTE[NUM_WAYPOINTS] = {{
    {c_array}
}};
"""

with open("5x5_route.h", "w") as f:
    f.write(c_header)
    
print("\nExported to '5x5_route.h'!")

In [ ]:
# Generate the string array (Passing waypoints and maze into the function)
master_mission = build_master_mission(
    START,
    ZONE_ENTER,
    ZONE_EXIT,
    GOAL,
    START_HEADING,
    waypoints,
    maze
)

c_array = ",\n    ".join(master_mission)

# Write to file
c_header = f"""#pragma once
// Master Mission Profile
// Contains discrete grid commands and continuous waypoints.

struct Command {{
    char action;
    float value;
}};

constexpr int NUM_COMMANDS = {len(master_mission)};
constexpr Command MISSION[NUM_COMMANDS] = {{
    {c_array}
}};
"""

with open("4_2_full_route.h", "w") as f:
    f.write(c_header)

print("Mission Profile successfully exported to '4_2_full_route.h'!")